In [0]:
import uuid

def get_batch_id() -> str:
    """
    Retrieves the Job Run ID from Databricks Workflows.
    If the run is manual (job_run_id is '0' or empty), generates a unique UUID 
    prefixed with the user's name for traceability.
    """
    # 1. Define and get the widget value (passed from Workflow as {{job.run_id}})
    dbutils.widgets.text("job_run_id", "0")
    batch_id = dbutils.widgets.get("job_run_id") 
    
    return batch_id

In [0]:
from pyspark.sql.functions import current_timestamp, col

def add_bronze_metadata(df):
    """
    Standardized lineage for Bronze (Unity Catalog Compliant).
    Extracts file path from the UC-native _metadata struct.
    """
    return df.withColumn("_ingested_at", current_timestamp()).withColumn(
        "_source_file", col("_metadata.file_path")
    ) 

In [0]:
from pyspark.sql.functions import current_timestamp, col

def apply_silver_metadata(df):
    """
    Injects Silver update timestamps and purges transient Bronze/CDF metadata.
    """
    transient_cols = [
        "_commit_version",
        "_commit_timestamp",
        "_rescued_data"
    ]

    # Preserve the Bronze commit version for Silver lineage fencing
    if "_commit_version" in df.columns:
        df = df.withColumn("_bronze_version", col("_commit_version"))

    # Append execution timestamp and purge transient columns via unpack
    return df.withColumn("_updated_at", current_timestamp()).drop(*transient_cols)

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, lit

def add_ref_metadata(df):
    """
    Standardizes technical metadata for reference (lookup) tables.
    """
    return (df
        .withColumn("_updated_at", current_timestamp())        
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_processed_by", lit("lending_risk_ingest_job"))
    )

In [0]:
from pyspark.sql.functions import current_timestamp, lit

def apply_gold_metadata(df, module_name):
    """
    Applies business-level lineage for Gold scoring tables.
    """
    return (df
        .withColumn("_updated_at", current_timestamp())
        .withColumn("_scoring_module", lit(module_name))
    )